In [ ]:
# ============================================================
# 0) Install

# ============================================================
!pip -q install -U transformers datasets huggingface_hub accelerate soundfile librosa jiwer tqdm pandas



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which i

In [ ]:
import os
import re
import unicodedata
from dataclasses import dataclass
from typing import Callable, Dict, Optional, List, Tuple

import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset, Audio

from jiwer import wer

In [ ]:

# ============================================================
# 1) Config
# ============================================================
import os
DATASET_ID = "AhmedAshrafMarzouk/bulbul-ps"
OUT_DIR = "./asr_runs"
os.makedirs(OUT_DIR, exist_ok=True)

MODELS = {
    # Whisper (enc-dec)
    "openai/whisper-large-v3": "whisper",
    "openai/whisper-large-v3-turbo": "whisper",

    # CTC (enc-only)
    "facebook/mms-1b-all": "ctc",

    # Seamless (speech-to-text)
    "facebook/seamless-m4t-v2-large": "seamless",
}

# Choose ONE model for the experiment run:
MODEL_LIST = [
"openai/whisper-large-v3",
"openai/whisper-large-v3-turbo",
"facebook/mms-1b-all",
"facebook/seamless-m4t-v2-large"
]

# Performance knobs
MAX_SAMPLES_PER_SPLIT = None  # e.g., 100 for a quick run; None = full split
SHOW_EXAMPLES = 3             # print a few examples per split



In [ ]:
# ============================================================
# 2) Arabic normalization (for scoring)
# ============================================================
ARABIC_BLOCKS = r"\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF"

def normalize_ar(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u0640", "")  # tatweel
    # remove combining marks (diacritics)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    # Arabic-Indic numbers -> ASCII
    for i, d in enumerate("٠١٢٣٤٥٦٧٨٩"):
        text = text.replace(d, str(i))
    # Alef variants
    text = re.sub(r"[أإآ]", "ا", text)
    # remove punctuation/symbols (keep Arabic + ascii word chars + whitespace)
    text = re.sub(rf"[^{ARABIC_BLOCKS}\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ============================================================
# 2.5) Metrics + Reporting (WER, CER, NWER, NCER) + Report CSV
# ============================================================
def _tokenize_words(s: str):
    s = (s or "").strip()
    return s.split() if s else []

def _tokenize_chars(s: str):
    # CER typically ignores spaces
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace(" ", "")
    return list(s) if s else []

def _edit_distance(ref_tokens, hyp_tokens) -> int:
    n, m = len(ref_tokens), len(hyp_tokens)
    if n == 0:
        return m
    if m == 0:
        return n

    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, m + 1):
            cur = dp[j]
            cost = 0 if ref_tokens[i - 1] == hyp_tokens[j - 1] else 1
            dp[j] = min(
                dp[j] + 1,      # deletion
                dp[j - 1] + 1,  # insertion
                prev + cost     # substitution
            )
            prev = cur
    return dp[m]

def _error_rate(refs, hyps, tokenizer) -> float:
    total_edits = 0
    total_ref_tokens = 0

    for r, h in zip(refs, hyps):
        rt = tokenizer(r)
        ht = tokenizer(h)
        total_edits += _edit_distance(rt, ht)
        total_ref_tokens += len(rt)

    if total_ref_tokens == 0:
        return 0.0 if all((not (h or "").strip()) for h in hyps) else 1.0
    return total_edits / total_ref_tokens

def compute_metrics(refs, hyps):
    # Raw
    WER = _error_rate(refs, hyps, _tokenize_words)
    CER = _error_rate(refs, hyps, _tokenize_chars)

    # Normalized
    refs_n = [normalize_ar(r) for r in refs]
    hyps_n = [normalize_ar(h) for h in hyps]
    NWER = _error_rate(refs_n, hyps_n, _tokenize_words)
    NCER = _error_rate(refs_n, hyps_n, _tokenize_chars)

    return {
        "WER": float(WER),
        "CER": float(CER),
        "NWER": float(NWER),
        "NCER": float(NCER),
    }

def append_report_row(report_path: str, row: dict):
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    if os.path.exists(report_path):
        df = pd.read_csv(report_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])
    df.to_csv(report_path, index=False, encoding="utf-8")


In [ ]:

# ============================================================
# 3) Model builders (Whisper / CTC / Seamless)
# ============================================================
def make_whisper_transcriber(model_id: str, language: str = "ar") -> Callable:
    from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=dtype, low_cpu_mem_usage=True
    ).to(device)
    model.eval()

    @torch.inference_mode()
    def transcribe(audio_array, sampling_rate: int = 16000) -> str:
        inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt")
        input_features = inputs.input_features.to(device, dtype=dtype)
        attention_mask = inputs.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        pred_ids = model.generate(
            input_features,
            attention_mask=attention_mask,
            language=language,
            task="transcribe",
            num_beams=1,
            do_sample=False,
        )
        return processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()

    return transcribe

from typing import Callable
import torch
from transformers import AutoProcessor, AutoModelForCTC

def make_ctc_transcriber(model_id: str, target_lang: str = "ara") -> Callable:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForCTC.from_pretrained(
        model_id, torch_dtype=dtype, low_cpu_mem_usage=True
    ).to(device)
    model.eval()

    # --- MMS-specific: pick language + load adapter ---
    processor.tokenizer.set_target_lang(target_lang)
    model.load_adapter(target_lang)

    @torch.inference_mode()
    def transcribe(audio_array, sampling_rate: int = 16000) -> str:
        inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt")

        input_values = inputs.input_values.to(device=device, dtype=dtype)
        attention_mask = inputs.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device=device)

        logits = model(input_values, attention_mask=attention_mask).logits
        pred_ids = torch.argmax(logits, dim=-1)
        return processor.batch_decode(pred_ids)[0].strip()

    return transcribe


def make_seamless_v2_asr(
    model_id: str = "facebook/seamless-m4t-v2-large",
    tgt_lang: str = "arb",
) -> Callable:
    import inspect
    from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    processor = AutoProcessor.from_pretrained(model_id)
    model = SeamlessM4Tv2ForSpeechToText.from_pretrained(model_id, torch_dtype=dtype).to(device)
    model.eval()

    # compat: some versions use audio=, others audios=
    proc_params = set(inspect.signature(processor.__call__).parameters.keys())
    audio_kw = "audio" if "audio" in proc_params else "audios"

    @torch.inference_mode()
    def transcribe(audio_array, sampling_rate: int = 16000, max_new_tokens: int = 256) -> str:
        wav = torch.tensor(audio_array)
        if wav.ndim == 2:
            wav = wav.mean(dim=0)
        wav = wav.to(torch.float32)

        if sampling_rate != 16000:
            raise ValueError("SeamlessM4T-v2 expects 16kHz; cast dataset audio to 16kHz.")

        inputs = processor(**{audio_kw: wav}, sampling_rate=16000, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        out_ids = model.generate(**inputs, tgt_lang=tgt_lang, max_new_tokens=max_new_tokens)
        return processor.batch_decode(out_ids, skip_special_tokens=True)[0].strip()

    return transcribe


def make_transcriber(model_id: str) -> Callable:
    fam = MODELS[model_id]
    if fam == "whisper":
        return make_whisper_transcriber(model_id, language="ar")
    if fam == "ctc":
        return make_ctc_transcriber(model_id, target_lang="ara")
    if fam == "seamless":
        return make_seamless_v2_asr(model_id, tgt_lang="arb")
    raise ValueError(f"Unknown family: {fam}")


In [ ]:

# ============================================================
# 4) Dataset utilities
# ============================================================
def pick_splits(ds_dict) -> Dict[str, str]:
    """
    Returns mapping like {"dev": "<actual_split_name>", "test": "<actual_split_name>"}.
    Prefers dev/validation then test.
    """
    names = list(ds_dict.keys())

    def first_existing(cands: List[str]) -> Optional[str]:
        for c in cands:
            if c in ds_dict:
                return c
        return None

    dev_name = first_existing(["dev", "validation", "valid", "val"])
    test_name = first_existing(["test", "eval", "evaluation"])

    # If dataset only has train/test, treat train as dev for convenience (but you can change this)
    if dev_name is None and "train" in ds_dict:
        dev_name = "train"
    if test_name is None and "test" in ds_dict:
        test_name = "test"

    out = {}
    if dev_name is not None:
        out["dev"] = dev_name
    if test_name is not None:
        out["test"] = test_name

    if not out:
        raise ValueError(f"Could not find dev/test-like splits in: {names}")
    return out


def get_audio_file(ex: dict) -> str:
    # Prefer your explicit column if present
    if "audio_file" in ex and ex["audio_file"] is not None:
        return str(ex["audio_file"])
    # Otherwise try audio metadata
    audio = ex.get("audio", {})
    if isinstance(audio, dict):
        if audio.get("path"):
            return str(audio["path"])
    return ""



In [ ]:

# ============================================================
# 5) Core experiment loop
# ============================================================
@dataclass
class SplitResult:
    split: str
    n: int
    WER: float
    CER: float
    NWER: float
    NCER: float
    csv_path: str

def run_split(
    ds_split,
    split_name: str,
    transcribe_fn: Callable,
    model_id: str,
    report_path: str,
    max_samples: Optional[int] = None,
) -> SplitResult:
    if max_samples is not None:
        ds_split = ds_split.select(range(min(max_samples, len(ds_split))))

    rows = []
    refs = []
    hyps = []

    pbar = tqdm(range(len(ds_split)), desc=f"{split_name} | {model_id}", leave=False)
    for i in pbar:
        ex = ds_split[i]
        audio = ex["Audio"]["array"]
        sr = ex["Audio"]["sampling_rate"]
        ref = ex.get("text", "")

        hyp = transcribe_fn(audio, sampling_rate=sr)

        refs.append(ref)
        hyps.append(hyp)

        rows.append({
            "audio_file": get_audio_file(ex),
            "text": ref,
            "predicted": hyp,
        })

        if i < SHOW_EXAMPLES:
            pbar.write(f"[{split_name} ex {i}]")
            pbar.write(f"  REF : {ref}")
            pbar.write(f"  HYP : {hyp}")
            pbar.write(f"  NREF: {normalize_ar(ref)}")
            pbar.write(f"  NHYP: {normalize_ar(hyp)}")

    # Save predictions CSV
    df = pd.DataFrame(rows)
    csv_path = os.path.join(OUT_DIR, f"preds_{split_name}_{model_id.replace('/', '_')}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8")

    # Compute metrics
    m = compute_metrics(refs, hyps)

    # Append a row to the global report
    append_report_row(report_path, {
        "model_id": model_id,
        "split": split_name,
        "n": len(df),
        **m,
        "preds_csv": csv_path,
    })

    return SplitResult(
        split=split_name,
        n=len(df),
        WER=m["WER"],
        CER=m["CER"],
        NWER=m["NWER"],
        NCER=m["NCER"],
        csv_path=csv_path
    )
def run_experiment(dataset_id: str, model_id: str) -> pd.DataFrame:
    ds = load_dataset(dataset_id)
    ds = ds.cast_column("Audio", Audio(sampling_rate=16000))

    split_map = pick_splits(ds)

    print("Loaded dataset splits:", list(ds.keys()))
    print("Using splits:", split_map)

    transcribe_fn = make_transcriber(model_id)

    report_path = os.path.join(OUT_DIR, "report.csv")

    results: List[SplitResult] = []
    for logical_name, actual_name in split_map.items():
        res = run_split(
            ds[actual_name],
            split_name=logical_name,
            transcribe_fn=transcribe_fn,
            model_id=model_id,
            report_path=report_path,
            max_samples=MAX_SAMPLES_PER_SPLIT,
        )
        results.append(res)

    out = pd.DataFrame([{
        "model_id": model_id,
        "split": r.split,
        "n": r.n,
        "WER": r.WER,
        "CER": r.CER,
        "NWER": r.NWER,
        "NCER": r.NCER,
        "csv_path": r.csv_path,
    } for r in results]).sort_values("split")

    return out

In [ ]:
# ============================================================
# 6) Run
# ============================================================
MODEL_ID = MODEL_LIST[0]  # <-- change to any key above

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "| VRAM(GB):", round(props.total_memory/1e9, 1))

results_df = run_experiment(DATASET_ID, MODEL_ID)
print("\n=== Final results ===")
display(results_df)

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB | VRAM(GB): 85.1


README.md:   0%|          | 0.00/481 [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/329M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/382M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/626 [00:00<?, ? examples/s]

...

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

[dev ex 0]
  REF : أختي زينب إن شاء الله نرجعهولك في فرحة لبنات
  HYP : أختي زينة بن صلى رجعوهولك في فرحة البنات
  NREF: اختي زينب ان شاء الله نرجعهولك في فرحة لبنات
  NHYP: اختي زينة بن صلى رجعوهولك في فرحة البنات
[dev ex 1]
  REF : الفقر و الفرعنة و المضرح
  HYP : الفقر والفرعنة والمظرح
  NREF: الفقر و الفرعنة و المضرح
  NHYP: الفقر والفرعنة والمظرح
[dev ex 2]
  REF : تربثت اقسم بالله لبني العرق
  HYP : تربثة أقسم بالله لبني العرق
  NREF: تربثت اقسم بالله لبني العرق
  NHYP: تربثة اقسم بالله لبني العرق


test | openai/whisper-large-v3:   0%|          | 0/744 [00:00<?, ?it/s]

[test ex 0]
  REF : الزهو و الطرب و المرض العضال الي اسمو الأمل
  HYP : الزهو وطرب والمرض العذار اللي اسمه الأمل
  NREF: الزهو و الطرب و المرض العضال الي اسمو الامل
  NHYP: الزهو وطرب والمرض العذار اللي اسمه الامل
[test ex 1]
  REF : ما احلاهم و ما اعزهم
  HYP : محلهم ومعظم
  NREF: ما احلاهم و ما اعزهم
  NHYP: محلهم ومعظم
[test ex 2]
  REF : باهي وشبيني كيف نحكي على الطلياني الي يلبس القمراية ويطيب مقرونة بالحبق ما تبعثليش
  HYP : باهي وش بيني كيف نحكي على الطالياني اللي يلبس القامراية وطيب مقرونة بالحبق ما تبعثليش
  NREF: باهي وشبيني كيف نحكي على الطلياني الي يلبس القمراية ويطيب مقرونة بالحبق ما تبعثليش
  NHYP: باهي وش بيني كيف نحكي على الطالياني اللي يلبس القامراية وطيب مقرونة بالحبق ما تبعثليش

=== Final results ===


,model_id,split,n,WER,CER,NWER,NCER,csv_path
0,openai/whisper-large-v3,dev,626,0.544423,0.145378,0.514020,0.131854,./asr_runs/preds_dev_openai_whisper-large-v3.csv
1,openai/whisper-large-v3,test,744,0.520294,0.131770,0.483622,0.117080,./asr_runs/preds_test_openai_whisper-large-v3.csv


In [ ]:

# ============================================================
# 6) Run
# ============================================================
MODEL_ID = MODEL_LIST[1]  # <-- change to any key above

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "| VRAM(GB):", round(props.total_memory/1e9, 1))

results_df = run_experiment(DATASET_ID, MODEL_ID)
print("\n=== Final results ===")
display(results_df)

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB | VRAM(GB): 85.1
Loaded dataset splits: ['dev', 'test']
Using splits: {'dev': 'dev', 'test': 'test'}


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

dev | openai/whisper-large-v3-turbo:   0%|          | 0/626 [00:00<?, ?it/s]

[dev ex 0]
  REF : أختي زينب إن شاء الله نرجعهولك في فرحة لبنات
  HYP : أكت زينبن صالرات رو هوليك في فرهيط ليت
  NREF: اختي زينب ان شاء الله نرجعهولك في فرحة لبنات
  NHYP: اكت زينبن صالرات رو هوليك في فرهيط ليت
[dev ex 1]
  REF : الفقر و الفرعنة و المضرح
  HYP : إل فقر و إلفارعنا و إلمادره
  NREF: الفقر و الفرعنة و المضرح
  NHYP: ال فقر و الفارعنا و المادره
[dev ex 2]
  REF : تربثت اقسم بالله لبني العرق
  HYP : تربث وتقسم بالله لبني العرق
  NREF: تربثت اقسم بالله لبني العرق
  NHYP: تربث وتقسم بالله لبني العرق


test | openai/whisper-large-v3-turbo:   0%|          | 0/744 [00:00<?, ?it/s]

[test ex 0]
  REF : الزهو و الطرب و المرض العضال الي اسمو الأمل
  HYP : الزهو وطرب والمرض العذال اللي اسمه الأمل
  NREF: الزهو و الطرب و المرض العضال الي اسمو الامل
  NHYP: الزهو وطرب والمرض العذال اللي اسمه الامل
[test ex 1]
  REF : ما احلاهم و ما اعزهم
  HYP : محليهم ومع الزن
  NREF: ما احلاهم و ما اعزهم
  NHYP: محليهم ومع الزن
[test ex 2]
  REF : باهي وشبيني كيف نحكي على الطلياني الي يلبس القمراية ويطيب مقرونة بالحبق ما تبعثليش
  HYP : بيهي وشبيني كيف نحكي على الطليين اللي يلبس القمرايا وطيب مقرونة بالحباق ما تبعث ليش
  NREF: باهي وشبيني كيف نحكي على الطلياني الي يلبس القمراية ويطيب مقرونة بالحبق ما تبعثليش
  NHYP: بيهي وشبيني كيف نحكي على الطليين اللي يلبس القمرايا وطيب مقرونة بالحباق ما تبعث ليش

=== Final results ===


,model_id,split,n,WER,CER,NWER,NCER,csv_path
0,openai/whisper-large-v3-turbo,dev,626,0.633269,0.204486,0.606846,0.189811,./asr_runs/preds_dev_openai_whisper-large-v3-t...
1,openai/whisper-large-v3-turbo,test,744,0.547140,0.142192,0.511561,0.127270,./asr_runs/preds_test_openai_whisper-large-v3-...


In [ ]:

# ============================================================
# 6) Run
# ============================================================
MODEL_ID = MODEL_LIST[2]  # <-- change to any key above

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "| VRAM(GB):", round(props.total_memory/1e9, 1))

results_df = run_experiment(DATASET_ID, MODEL_ID)
print("\n=== Final results ===")
display(results_df)

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB | VRAM(GB): 85.1
Loaded dataset splits: ['dev', 'test']
Using splits: {'dev': 'dev', 'test': 'test'}


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

dev | facebook/mms-1b-all:   0%|          | 0/626 [00:00<?, ?it/s]

[dev ex 0]
  REF : أختي زينب إن شاء الله نرجعهولك في فرحة لبنات
  HYP : أقتزي بنصلرترهولك في فرحة لابنات
  NREF: اختي زينب ان شاء الله نرجعهولك في فرحة لبنات
  NHYP: اقتزي بنصلرترهولك في فرحة لابنات
[dev ex 1]
  REF : الفقر و الفرعنة و المضرح
  HYP : الفقر والفرعن والمظرح
  NREF: الفقر و الفرعنة و المضرح
  NHYP: الفقر والفرعن والمظرح
[dev ex 2]
  REF : تربثت اقسم بالله لبني العرق
  HYP : تربثت أقص بالله لابني العرق
  NREF: تربثت اقسم بالله لبني العرق
  NHYP: تربثت اقص بالله لابني العرق


test | facebook/mms-1b-all:   0%|          | 0/744 [00:00<?, ?it/s]

[test ex 0]
  REF : الزهو و الطرب و المرض العضال الي اسمو الأمل
  HYP : اذه وطرب المارس العظال إلي اسم الأمل
  NREF: الزهو و الطرب و المرض العضال الي اسمو الامل
  NHYP: اذه وطرب المارس العظال الي اسم الامل
[test ex 1]
  REF : ما احلاهم و ما اعزهم
  HYP : محليهم مع الزوم
  NREF: ما احلاهم و ما اعزهم
  NHYP: محليهم مع الزوم
[test ex 2]
  REF : باهي وشبيني كيف نحكي على الطلياني الي يلبس القمراية ويطيب مقرونة بالحبق ما تبعثليش
  HYP : به وشبني كيف نحك على الطليان إلي يلبس القمراية وطيب مقرن بالحضق متبعث ليش
  NREF: باهي وشبيني كيف نحكي على الطلياني الي يلبس القمراية ويطيب مقرونة بالحبق ما تبعثليش
  NHYP: به وشبني كيف نحك على الطليان الي يلبس القمراية وطيب مقرن بالحضق متبعث ليش

=== Final results ===


,model_id,split,n,WER,CER,NWER,NCER,csv_path
0,facebook/mms-1b-all,dev,626,0.746731,0.219836,0.726745,0.206713,./asr_runs/preds_dev_facebook_mms-1b-all.csv
1,facebook/mms-1b-all,test,744,0.732343,0.202550,0.707129,0.188010,./asr_runs/preds_test_facebook_mms-1b-all.csv


In [ ]:

# ============================================================
# 6) Run
# ============================================================
MODEL_ID = MODEL_LIST[3]  # <-- change to any key above

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "| VRAM(GB):", round(props.total_memory/1e9, 1))

results_df = run_experiment(DATASET_ID, MODEL_ID)
print("\n=== Final results ===")
display(results_df)

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB | VRAM(GB): 85.1
Loaded dataset splits: ['dev', 'test']
Using splits: {'dev': 'dev', 'test': 'test'}


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

...

dev | facebook/seamless-m4t-v2-large:   0%|          | 0/626 [00:00<?, ?it/s]

[dev ex 0]
  REF : أختي زينب إن شاء الله نرجعهولك في فرحة لبنات
  HYP : أختي زينب بن صل الله الرجولك في فرحة البنات.
  NREF: اختي زينب ان شاء الله نرجعهولك في فرحة لبنات
  NHYP: اختي زينب بن صل الله الرجولك في فرحة البنات
[dev ex 1]
  REF : الفقر و الفرعنة و المضرح
  HYP : الفقر والفرعنة والمذبح
  NREF: الفقر و الفرعنة و المضرح
  NHYP: الفقر والفرعنة والمذبح
[dev ex 2]
  REF : تربثت اقسم بالله لبني العرق
  HYP : تربت أقسم بالله لابني العرق
  NREF: تربثت اقسم بالله لبني العرق
  NHYP: تربت اقسم بالله لابني العرق


test | facebook/seamless-m4t-v2-large:   0%|          | 0/744 [00:00<?, ?it/s]

[test ex 0]
  REF : الزهو و الطرب و المرض العضال الي اسمو الأمل
  HYP : الزهو والطرب والمرض العزال اللي اسمه الأمل.
  NREF: الزهو و الطرب و المرض العضال الي اسمو الامل
  NHYP: الزهو والطرب والمرض العزال اللي اسمه الامل
[test ex 1]
  REF : ما احلاهم و ما اعزهم
  HYP : محليهم ومعظمهم
  NREF: ما احلاهم و ما اعزهم
  NHYP: محليهم ومعظمهم
[test ex 2]
  REF : باهي وشبيني كيف نحكي على الطلياني الي يلبس القمراية ويطيب مقرونة بالحبق ما تبعثليش
  HYP : باهي وشبيني كيف نحكي على الإيطالي الي يلبس القمرية ويطيب مقرونة بالحبق ما تبعثليش
  NREF: باهي وشبيني كيف نحكي على الطلياني الي يلبس القمراية ويطيب مقرونة بالحبق ما تبعثليش
  NHYP: باهي وشبيني كيف نحكي على الايطالي الي يلبس القمرية ويطيب مقرونة بالحبق ما تبعثليش

=== Final results ===


,model_id,split,n,WER,CER,NWER,NCER,csv_path
0,facebook/seamless-m4t-v2-large,dev,626,0.548269,0.156473,0.488494,0.129742,./asr_runs/preds_dev_facebook_seamless-m4t-v2-...
1,facebook/seamless-m4t-v2-large,test,744,0.518057,0.133586,0.452152,0.106217,./asr_runs/preds_test_facebook_seamless-m4t-v2...
